# 🍎 Notebook 1 — Introduction to embeddings

### The problem Catherine needs us to help with
We have two big lists of foods. One list uses everyday names (**aubergine**, **chips**, **cuppa**). The other uses long official names (**Eggplant, cooked, boiled, drained**). We want a computer to automatically match each everyday name to the right official one.

That sounds easy until you realise: **a computer has no idea what food is.** It doesn't know an aubergine is an eggplant. To it, they're just different strings of letters. This notebook is about giving the computer a way to compare *meaning* instead of spelling.

| Part | What you'll learn | Roughly |
|---|---|---|
| **1** | A quick Python warm-up | 15 min |
| **2** | The big idea: turning meaning into numbers | 25 min |
| **3** | Measuring similarity — distance and *cosine* | 30 min |
| ☕ | *good place for a break* | |
| **4** | Real embeddings from an AI model | 20 min |
| **5** | Seeing the whole "map" of food | 15 min |
| **6** | Searching, and the number `k` | 20 min |
| **7** | Checking if it actually works | 15 min |

> **No coding experience needed.** Run each cell with **Shift + Enter** and *read the output before moving on*. Every code cell has plain-English comments. The bits marked **🎯 YOUR TURN** are for you to change and experiment with.


### Run this first — it sets everything up
Takes about 30 seconds. It installs the tools and downloads a small AI model we'll use later. You'll see a lot of text scroll past — that's normal.


In [1]:
!pip -q install sentence-transformers
import numpy as np              # numbers and maths
import pandas as pd             # tables
import matplotlib.pyplot as plt # charts and diagrams
print("\n✅ Ready to go.")



✅ Ready to go.


---
## Part 1 · Quick Python recap

Before the AI, three tiny bits of Python that we'll use over and over. If you did the Python intro already, this is just a refresher — run the cells and move on.


### 1.1 · Variables — giving a name to a value
A **variable** is an object that stores something. We put a value in with `=`, and we can use the label later.


In [ ]:
food = "aubergine"     # a piece of text (a "string") — note the quotes
price = 1.20            # a number
print(food)
print("It costs", price, "pounds")


### 1.2 · Lists — a group of values
A **list** holds several values in order, inside square brackets. We can ask how long it is, and pull out items by their position. (Python counts from **0**, so the first item is item `0`.)


In [ ]:
foods = ["apple", "banana", "cheese", "bacon"]

print("The whole list:", foods)
print("How many:", len(foods))
print("The first one:", foods[0])
print("The third one:", foods[2])


### 1.3 · Functions — a reusable machine
A **function** is a mini-machine: you give it something (an *input*), it does some work, and it hands something back (a *return* value). We'll write functions so we don't repeat ourselves.


In [ ]:
def shout(word):          # 'word' is the input
    return word.upper() + "!"   # turns whatever word you pass it uppercase, hand back the result

print(shout("hello"))
print(shout("bananas"))


> **✅ Checkpoint.** You now know variables, lists and functions. That's genuinely most of the Python we need. On to the interesting part.


---
## Part 2 · The big idea: turning meaning into numbers

### 2.1 · Why a computer can't compare food
Let's prove the problem. Ask Python whether two words are the same:


In [ ]:
word_1 = "aubergine"
word_2 = "eggplant"
print(word_1 == word_2)


It says `False`. The `==` compared the **letters**, one by one, and they're different. The computer has no idea these are the same vegetable.

To fix this, we need to describe each food in a way a computer *can* compare — and computers are brilliant at one thing: **numbers**.


### 2.2 · Describing a food with numbers
Here's the trick. Imagine we score every food on two things, each from 0 to 10:

- **sweetness** — how sweet is it?
- **savouriness** — how savoury (salty / meaty) is it?

So an apple might be `[8, 1]` (very sweet, not savoury) and bacon might be `[1, 9]` (not sweet, very savoury). Each food becomes a little list of two numbers. Let's write some down.


In [ ]:
# Each food gets [sweetness, savouriness], scored 0-10 by hand.
food_scores = {
    "apple":      [8, 1],
    "banana":     [7, 1],
    "honey":      [10, 0],
    "chocolate":  [9, 2],
    "bread":      [2, 4],
    "cheddar":    [1, 7],
    "bacon":      [1, 9],
    "sausage":    [1, 8],
    "crisps":     [2, 7],
    "chicken":    [1, 6],
}

# Put it in a table so it's easy to read.
table = pd.DataFrame(food_scores, index=["sweetness", "savouriness"]).T
table


### 2.3 · Drawing our foods on a map
Because each food is just two numbers, we can treat them as **coordinates** and plot every food as a dot. Sweetness goes across (x), savouriness goes up (y).

Watch what happens to foods that are alike.


In [ ]:
plt.figure(figsize=(8, 8))
for name, (sweet, savoury) in food_scores.items():
    plt.scatter(sweet, savoury, s=120)
    plt.text(sweet + 0.15, savoury + 0.15, name, fontsize=11)

plt.xlabel("sweetness  →"); plt.ylabel("savouriness  →")
plt.title("Our hand-made map of food")
plt.xlim(-1, 11); plt.ylim(-1, 11); plt.grid(alpha=0.3)
plt.show()


### 2.4 · The key insight
Look at the map. The **sweet things** (apple, banana, honey, chocolate) cluster in the bottom-right. The **savoury things** (bacon, sausage, cheddar, crisps) cluster up the left. Similar foods ended up **near each other**, and different foods ended up **far apart** — *just because we turned them into numbers.*

That list of numbers describing a thing is called an **embedding**. That's the whole idea. Everything else in this notebook is about doing this properly and at scale.

> An **embedding** is a list of numbers that represents the *meaning* of something, positioned so that similar things are close together.


### 2.5 · Two numbers isn't enough
Our map can't tell apart two foods with the same scores. Bacon `[1,9]` and sausage `[1,8]` sit almost on top of each other, but they're not identical. And where would you put "fizzy" or "spicy" or "crunchy"?

Real problems need **more dimensions** — more numbers per food. We can't easily *draw* 5 or 300 numbers, but a computer does the maths in any number of dimensions just fine. The AI model we meet in Part 4 uses **384 numbers** for every food. First, though, we need a proper way to measure "closeness".


---
## Part 3 · Measuring similarity

We can *see* that apple and banana are close on our map. But the computer needs to **calculate** it. There are two natural ways to measure how alike two points are: **distance** and **direction**.


### 3.1 · Distance (the ruler method)
The obvious way: measure the straight-line distance between two dots, like a ruler. Small distance = similar. Let's measure apple-to-banana and apple-to-bacon.


In [ ]:
def distance(food_a, food_b):
    a = np.array(food_scores[food_a])   # e.g. [8, 1]
    b = np.array(food_scores[food_b])   # e.g. [7, 1]
    return np.sqrt(np.sum((a - b) ** 2))  # Pythagoras: √((x-x)² + (y-y)²)

print("apple ↔ banana:", round(distance("apple", "banana"), 2))
print("apple ↔ bacon :", round(distance("apple", "bacon"), 2))


Apple and banana are close (small number); apple and bacon are far (big number). That matches our intuition. So why not just use distance?

### 3.2 · Why distance can mislead
Distance is affected by **size** as well as direction. Imagine two portions of the same food: a tiny scoop `[1, 8]` and a huge platter `[2, 16]`. They're the *same food*, just more of it — but a ruler says they're far apart. What actually tells us they're the same is that they point in the **same direction** from the centre.

For meaning, **direction usually matters more than size.** That leads us to the measure that AI systems actually use: **cosine similarity**.


### 3.3 · Vectors and angles
From now on, picture each food not as a dot but as an **arrow** from the origin `(0,0)` out to its point. An arrow like this is called a **vector**. Two foods are similar if their arrows **point the same way** — if the *angle* between them is small.


In [ ]:
# Draw two foods as arrows and show the angle between them.
def arrow(ax, vec, label, color):
    ax.annotate("", xy=vec, xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.5))
    ax.text(vec[0]*1.05, vec[1]*1.05, label, color=color, fontsize=12)

a = np.array(food_scores["bacon"])     # [1, 9]
b = np.array(food_scores["sausage"])   # [1, 8]
c = np.array(food_scores["honey"])     # [10, 0]

fig, ax = plt.subplots(figsize=(7, 7))
arrow(ax, a, "bacon", "tab:red")
arrow(ax, b, "sausage", "tab:orange")
arrow(ax, c, "honey", "tab:green")
ax.set_xlim(0, 11); ax.set_ylim(0, 11); ax.grid(alpha=0.3)
ax.set_xlabel("sweetness →"); ax.set_ylabel("savouriness →")
ax.set_title("Foods as arrows (vectors).\nbacon & sausage point almost the same way; honey points elsewhere")
plt.show()


See how the **bacon** and **sausage** arrows lie almost on top of each other — a tiny angle between them — while **honey** points off in a completely different direction. Cosine similarity turns that angle into a single number.


### 3.4 · What "cosine" actually means
"Cosine" comes from trigonometry, but you don't need the maths to get the intuition. The **cosine of an angle** is just a number that describes how aligned two arrows are:

| Angle between arrows | They are... | Cosine |
|---|---|---|
| **0°** (pointing the same way) | as similar as possible | **1.0** |
| **90°** (at right angles) | unrelated | **0.0** |
| **180°** (opposite directions) | as different as possible | **−1.0** |

So cosine similarity always lands between −1 and 1: **near 1 = very similar, near 0 = unrelated.** Let's *see* those three cases.


In [ ]:
def show_case(ax, v1, v2, title):
    for v, col in [(v1, "tab:blue"), (v2, "tab:red")]:
        ax.annotate("", xy=v, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color=col, lw=2.5))
    cos = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    ax.set_title(f"{title}\ncosine = {cos:.2f}")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
    ax.set_aspect("equal"); ax.grid(alpha=0.3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
show_case(axes[0], [2, 1],  [2, 1.05], "Same direction")
show_case(axes[1], [2, 1],  [-1, 2],   "Right angle")
show_case(axes[2], [2, 1],  [-2, -1],  "Opposite")
plt.tight_layout(); plt.show()


### 3.5 · Computing cosine from the numbers
Here's the actual recipe (it's short). To compare two number-lists `a` and `b`:

1. **Multiply them together position-by-position and add it all up.** This is called the *dot product*. It's big when the arrows point the same way.
2. **Divide by the length of each arrow** (its "magnitude"), to cancel out the size effect we worried about in 3.2.

That's it. In code:


In [ ]:
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    dot = np.sum(a * b)                       # step 1: dot product
    lengths = np.linalg.norm(a) * np.linalg.norm(b)  # step 2: the two arrow lengths
    return dot / lengths

print("bacon vs sausage:", round(cosine(food_scores["bacon"],  food_scores["sausage"]), 3))
print("bacon vs honey  :", round(cosine(food_scores["bacon"],  food_scores["honey"]),   3))
print("apple vs banana :", round(cosine(food_scores["apple"],  food_scores["banana"]),  3))


bacon vs sausage is near **1** (almost the same). bacon vs honey is low. This one little function is the heart of the whole matching system — and it works exactly the same whether a food has 2 numbers or 384.


### 🎯 YOUR TURN
Change the two foods below and run it. Try to find:
1. the **most similar** pair you can (closest to 1.0);
2. the **least similar** pair.

Foods available: `apple, banana, honey, chocolate, bread, cheddar, bacon, sausage, crisps, chicken`.


In [ ]:
FOOD_A = "chocolate"
FOOD_B = "cheddar"

print(f"cosine similarity of {FOOD_A} and {FOOD_B}:",
      round(cosine(food_scores[FOOD_A], food_scores[FOOD_B]), 3))


> **☕ Good place for a break.** You've learned the single most important idea in the whole week: *meaning becomes numbers, and cosine measures how aligned those numbers are.* After the break, we swap our hand-made scores for a real AI model.


---
## Part 4 · Real embeddings from an AI model

### 4.1 · The problem with doing it by hand
We scored 10 foods on 2 things. But there are **thousands** of foods, and "sweetness/savouriness" can't capture everything (texture, temperature, ingredients, whether it's a drink…). Nobody could hand-score all that.

So instead we use an **embedding model**: an AI that has read enormous amounts of text and *learned by itself* how to turn any word or phrase into a list of numbers — in our case, **384** numbers — that capture its meaning. We don't tell it what the numbers mean; it figured out useful ones during training.


In [ ]:
from sentence_transformers import SentenceTransformer

# Load a small, popular embedding model (downloads once, ~90 MB).
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

vec = embedder.encode("apple")
print("An embedding is a list of", len(vec), "numbers.")
print("The first 10 numbers for 'apple':")
print(np.round(vec[:10], 3))


### 4.2 · What do those numbers look like?
Unlike our tidy sweetness/savouriness, these 384 numbers aren't individually meaningful to us — no single one is "the sweetness number". The meaning is spread across all of them together. Here's the whole fingerprint for one food:


In [ ]:
vec = embedder.encode("apple")
plt.figure(figsize=(12, 3))
plt.bar(range(len(vec)), vec)
plt.title("The 384-number embedding for 'apple' (its 'meaning fingerprint')")
plt.xlabel("position (0-383)"); plt.ylabel("value")
plt.tight_layout(); plt.show()


### 4.3 · The same cosine function still works
This is the beautiful part: we don't need anything new. Our `cosine` function compares lists of numbers — it doesn't care if there are 2 or 384. Let's embed some foods and compare them.


In [ ]:
apple  = embedder.encode("apple")
pear   = embedder.encode("pear")
burger = embedder.encode("beef burger")

print("apple vs pear       :", round(cosine(apple, pear), 3))
print("apple vs beef burger:", round(cosine(apple, burger), 3))


apple and pear (both fruit) score high; apple and beef burger score lower. The model *taught itself* that apples and pears are alike — and it knows things our hand-made scores never could, like that "aubergine" and "eggplant" mean the same thing. Let's test exactly that:


In [ ]:
print("aubergine vs eggplant:", round(cosine(embedder.encode("aubergine"),
                                                embedder.encode("eggplant")), 3))
print("aubergine vs banana  :", round(cosine(embedder.encode("aubergine"),
                                                embedder.encode("banana")), 3))


### 🎯 YOUR TURN — the similarity game
Run the cell, type any two foods, and — **before you look** — say out loud what you think the score will be. Closest guess wins.

Try: two fruits · a fruit and a meat · a food and something that isn't food at all (`homework`, `dinosaur`).


In [ ]:
from ipywidgets import interact, Text

def compare(food_1, food_2):
    v1 = embedder.encode(food_1)
    v2 = embedder.encode(food_2)
    print(f"cosine similarity of {food_1!r} and {food_2!r}:  {cosine(v1, v2):.3f}")

interact(compare, food_1=Text("aubergine"), food_2=Text("eggplant"));


---
## Part 5 · Seeing the whole map of food

### 5.1 · A grid of every comparison
Let's embed a set of foods and compare **every food with every other food** at once. We'll draw it as a **heatmap**: bright squares = similar, dark squares = different. The bright diagonal is each food compared with itself (always 1.0).


In [ ]:
foods = [
    "apple", "pear", "banana", "orange",
    "aubergine", "eggplant", "potato", "carrot",
    "beef burger", "hamburger", "chicken breast", "salmon",
    "whole milk", "cheddar cheese",
    "orange juice", "fizzy drink", "coffee",
    "chips", "french fried potatoes",
]

# normalize_embeddings=True makes every arrow length 1, so cosine is just the dot product.
food_emb = embedder.encode(foods, normalize_embeddings=True)
similarity = food_emb @ food_emb.T    # every pair at once

plt.figure(figsize=(10, 9))
plt.imshow(similarity, cmap="viridis")
plt.colorbar(label="cosine similarity")
plt.xticks(range(len(foods)), foods, rotation=90)
plt.yticks(range(len(foods)), foods)
plt.title("Every food compared with every other food")
plt.tight_layout(); plt.show()


**Look for the bright blocks.** `aubergine`/`eggplant`, `beef burger`/`hamburger`, and `chips`/`french fried potatoes` light up — the model knows they're near-synonyms. The fruits form a bright square together, as do the meats.


### 5.2 · Flattening 384 numbers into a picture
A heatmap is useful but we still can't *see* the 384-dimensional space. There's a clever trick called **PCA** that squashes those 384 numbers down to just 2 — keeping as much of the structure as possible — so we can plot each food as a dot again, exactly like our hand-made map in Part 2, but now powered by the real model.


In [ ]:
from sklearn.decomposition import PCA

groups = (["fruit"]*4 + ["veg"]*4 + ["meat/fish"]*4 + ["dairy"]*2
          + ["drink"]*3 + ["prepared"]*2)
colours = {"fruit":"tab:red","veg":"tab:green","meat/fish":"tab:brown",
           "dairy":"tab:blue","drink":"tab:purple","prepared":"tab:orange"}

xy = PCA(n_components=2).fit_transform(food_emb)
plt.figure(figsize=(11, 8))
for (x, y), name, g in zip(xy, foods, groups):
    plt.scatter(x, y, s=110, color=colours[g])
    plt.text(x + 0.015, y + 0.015, name, fontsize=10)
plt.title("The model's map of food (384 numbers squashed to 2)\nNobody told it the groups — it worked them out from meaning")
plt.axis("off"); plt.tight_layout(); plt.show()


### Questions to discuss
1. Which foods landed right next to their synonym?
2. Do the drinks cluster together? The fruits?
3. Is anything in a surprising place — and can you guess why?
4. This is a *flattened* approximation. Why should we be a bit careful reading too much into exact positions?


---
## Part 6 · Searching, and the number `k`

### 6.1 · The actual task: find the closest match
Matching databases is just **search**: given a query food, rank every option by cosine similarity and return the closest ones. Returning the top few is called **retrieval**, and the number we return is called **`k`**.


In [ ]:
def retrieve(query, k=5):
    """Return the k foods most similar to `query`, best first."""
    q = embedder.encode(query, normalize_embeddings=True)
    scores = food_emb @ q                     # cosine with every food at once
    best = np.argsort(scores)[::-1][:k]       # the k highest scores
    return [(foods[i], round(float(scores[i]), 3)) for i in best]

for hit in retrieve("aubergine", k=5):
    print(hit)


It found `eggplant` at the top, even though the letters are completely different — because it's comparing *meaning*. This is the thing our very first `==` test could never do.

### 🎯 YOUR TURN — search anything
Type a query and drag the `k` slider. Try a word that **isn't** in the list at all (`low fat milk`, `soda`, `french fries`) and see whether the model still finds the right food.


In [ ]:
from ipywidgets import IntSlider

def search(query, k):
    for rank, (food, s) in enumerate(retrieve(query, k), 1):
        print(f"{rank:>2}.  {s:.3f}   {food}")

interact(search, query=Text("low fat milk"), k=IntSlider(5, 1, len(foods)));


### 6.2 · What `k` really controls
There are two different questions hiding inside a search:

1. **Did the right answer appear *anywhere* in the top `k`?** (retrieval)
2. **Was it *first*?** (ranking)

A **bigger `k`** makes it more likely the correct answer is *somewhere* in the list — good — but the list gets longer and has more wrong answers in it, which makes life harder for whatever picks the final winner. A **smaller `k`** gives a short, clean list but might chop off the right answer. Choosing `k` is a balance, and it's one of the things you'll optimise later this week. Watch the answer for `chips` move as `k` grows:


In [ ]:
print("Top 3 for 'chips':")
for h in retrieve("chips", 3): print("   ", h)
print("\nTop 8 for 'chips':")
for h in retrieve("chips", 8): print("   ", h)


### 6.3 · Where it goes wrong
Embeddings are powerful but not magic. Knowing how they fail is half the skill:

- **Ambiguous words.** British `chips` = fries; American `chips` = crisps; baking `chips` = chocolate. One word, three foods.
- **Missing detail.** `milk` doesn't say whole / skimmed / oat — but nutritionally those differ a lot.
- **Shared words, different food.** `orange juice` and `orange squash` both contain "orange" and score high, but aren't the same drink.

See that last trap for yourself:


In [ ]:
oj     = embedder.encode("orange juice", normalize_embeddings=True)
squash = embedder.encode("orange squash", normalize_embeddings=True)
water  = embedder.encode("sparkling water", normalize_embeddings=True)
print("orange juice vs orange squash :", round(float(oj @ squash), 3), " (high — shared word)")
print("orange juice vs sparkling water:", round(float(oj @ water), 3))
print("\nThe high score is partly the word 'orange', not the actual drink. This is exactly")
print("the kind of mistake a second AI step (in Notebook 3) can help catch.")


---
## Part 7 · Checking if it actually works

### 7.1 · You can't improve what you can't measure
How do we know if the search is *good*? We can't just eyeball it. We need a set of questions where a **human has already decided the right answer**, then count how often the model agrees. Below, five queries with their correct match. We measure whether the right answer is at rank 1, in the top 3, or in the top 5.


In [ ]:
test_cases = [
    ("aubergine",        "eggplant"),
    ("beef hamburger",   "hamburger"),
    ("fizzy pop",        "fizzy drink"),
    ("chicken fillet",   "chicken breast"),
    ("fried chips",      "french fried potatoes"),
]

rows = []
for query, correct in test_cases:
    ranked = [f for f, _ in retrieve(query, k=5)]
    rank = ranked.index(correct) + 1 if correct in ranked else None
    rows.append({"query": query, "correct answer": correct,
                 "model's top pick": ranked[0],
                 "rank of correct": rank,
                 "right at #1": rank == 1,
                 "in top 3": rank is not None and rank <= 3})
results = pd.DataFrame(rows)
display(results)
print(f"\nCorrect at rank 1: {100*results['right at #1'].mean():.0f}%   |   "
      f"In top 3: {100*results['in top 3'].mean():.0f}%")


Notice that "in top 3" usually beats "right at #1": the correct answer is often *in* the list, just not first. That's a big clue — a second step (a cleverer model, or a human) can re-pick the winner from a short list. **That second step is exactly what Notebook 3 is about.**

### 🎯 YOUR TURN
Add two of your own test cases to the list below (a query and the answer you think is correct), then run it. Did the model agree with you? Where did it disagree, and can you guess why?


In [ ]:
my_tests = [
    ("your query here",     "the answer you expect"),
    ("another query",       "its expected answer"),
]

for query, correct in my_tests:
    ranked = [f for f, _ in retrieve(query, k=5)]
    print(f"{query!r}: top pick = {ranked[0]!r}  |  full top 5 = {ranked}")


---
## ✅ What you learned

- A computer can't compare **meaning** directly — but it can compare **numbers**.
- An **embedding** is a list of numbers representing meaning, arranged so similar things sit close together.
- **Cosine similarity** measures how aligned two embeddings are: **1 = same, 0 = unrelated, −1 = opposite**.
- A real embedding model does this automatically with **384 numbers per food**, capturing things we could never score by hand (like aubergine = eggplant).
- **Searching** ranks options by similarity; **`k`** trades *catching* the answer against *cleanliness* of the shortlist.
- You can only tell if it's working by **checking against human answers**.

**Next → Notebook 2.** Some names are too short or too weird for the embedder alone (`cuppa`, `toad in the hole`). There, you'll meet a **language model** that can rewrite a tricky name into helpful words *before* it's embedded — and you'll see how it actually works under the hood.
